In [ ]:
!nvidia-smi

Fri May  1 09:43:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             31W /   70W |     357MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.quantization as quantization
import copy
import time
import numpy as np
import h5py
import pickle
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import os



In [ ]:
# ============================================
# MODEL ARCHITECTURE (from training)
# ============================================

class GRUClassifier(nn.Module):
    """original GRU model - this is the teacher"""
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True, dropout=0.3)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h = self.gru(x)
        return self.head(h[-1])


class StudentGRU(nn.Module):
    """Smaller student for distillation - dimensions match teacher's output"""
    def __init__(self, input_size=7, hidden_size=16, num_classes=4):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, 1, batch_first=True)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h = self.gru(x)
        return self.head(h[-1])

In [ ]:
# ============================================
# UTILITY FUNCTIONS FOR TRACKING
# ============================================

class ModelMetrics:
    """Track performance metrics throughout compression."""
    
    def __init__(self):
        self.stages = []
        self.sizes_kb = []
        self.accuracies = []
        self.inference_times_ms = []
        self.sparsities = []
        self.parameter_counts = []


    def log_stage(self, stage_name, model, test_loader, device):
        import io
        """Log metrics for current model stage."""
        print(f"\n{'='*60}")
        print(f"↝ Evaluating: {stage_name}")
        print(f"{'='*60}")
        
        # Determine actual device of model
        try:
            model_device = next(model.parameters()).device
        except:
            model_device = torch.device('cpu')
        
        # Calculate model size using serialization (works for any model, including quantized)
        # we used this because normal size calculations fails with quantization due to the way they store parameters 
        buffer = io.BytesIO()
        torch.save(model.state_dict(), buffer)
        size_bytes = buffer.tell()
        size_kb = size_bytes / 1024
        
        # Calculate sparsity (works only for models that still have parameters) not quantization that uses tensors instead of parameters
        total_params = 0
        zero_params = 0
        for param in model.parameters():
            total_params += param.numel()
            zero_params += (param == 0).sum().item()
        sparsity = (zero_params / total_params) * 100 if total_params > 0 else 0
        
        # Calculate inference time 
        model.eval()
        with torch.no_grad():
            # Warm-up
            for _ in range(5):
                for X_batch, _ in test_loader:
                    X_batch = X_batch.to(model_device)
                    _ = model(X_batch)
            # Measure
            start_time = time.time()
            total_samples = 0
            for X_batch, _ in test_loader:
                X_batch = X_batch.to(model_device)
                _ = model(X_batch)
                total_samples += len(X_batch)
            end_time = time.time()
            avg_time_ms = (end_time - start_time) / total_samples * 1000 if total_samples > 0 else 0
        
        # Calculate accuracy 
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(model_device)
                logits = model(X_batch)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y_batch.numpy())
        accuracy = accuracy_score(all_labels, all_preds) * 100
        
        # Store metrics
        param_count = sum(p.numel() for p in model.parameters())
        
        self.stages.append(stage_name)
        self.sizes_kb.append(size_kb)
        self.accuracies.append(accuracy)
        self.inference_times_ms.append(avg_time_ms)
        self.sparsities.append(sparsity)
        self.parameter_counts.append(param_count)
        
        # Print results
        print(f"\n ↝ Results for {stage_name}:")
        print(f"   Parameters:     {param_count:,}")
        print(f"   Model size:     {size_kb:.2f} KB ({size_kb/1024:.3f} MB)")
        print(f"   Sparsity:       {sparsity:.1f}%")
        print(f"   Accuracy:       {accuracy:.2f}%")
        print(f"   Inference time: {avg_time_ms:.3f} ms per sample")
        print(f"   Device:         {model_device}")
        
        return accuracy, size_kb, avg_time_ms    
        
    def print_summary_table(self):
        """Print a formatted summary table of all stages."""
        print("\n" + "="*90)
        print(" ↝ COMPRESSION PIPELINE SUMMARY")
        print("="*90)
        
        print(f"{'Stage':<20} {'Params':<12} {'Size(KB)':<12} {'Sparsity':<10} {'Accuracy':<12} {'Time(ms)':<12}")
        print("-"*90)
        
        for i, stage in enumerate(self.stages):
            print(f"{stage:<20} {self.parameter_counts[i]:<12,} {self.sizes_kb[i]:<12.2f} "
                  f"{self.sparsities[i]:<10.1f}% {self.accuracies[i]:<12.2f}% "
                  f"{self.inference_times_ms[i]:<12.3f}")
        
        print("-"*90)
        total_compression = (1 - self.sizes_kb[-1] / self.sizes_kb[0]) * 100
        accuracy_drop = self.accuracies[0] - self.accuracies[-1]
        
        print(f"\n ↝ FINAL RESULTS:")
        print(f"   Total Compression:  {total_compression:.1f}% smaller")
        print(f"   Accuracy Change:    {accuracy_drop:+.2f}%")
        print(f"   Final Model Size:   {self.sizes_kb[-1]:.2f} KB ({self.sizes_kb[-1]/1024:.3f} MB)")
        
        print(f"\n💡 RECOMMENDATION:")
        if accuracy_drop < 2:
            print(f" ✓   Excellent! Minimal accuracy loss ({accuracy_drop:.2f}%) with {total_compression:.0f}% compression.")
        elif accuracy_drop < 5:
            print(f"   🟢 Good. Acceptable accuracy trade-off for {total_compression:.0f}% compression.")
        elif accuracy_drop < 10:
            print(f"   🟡 Moderate accuracy loss. Consider using fewer compression stages.")
        else:
            print(f"   🔴 High accuracy loss. Revert to earlier stage or adjust hyperparameters.")
    
    def save_to_file(self, filepath):
        """Save metrics to a CSV file."""
        import pandas as pd
        df = pd.DataFrame({
            'Stage': self.stages,
            'Parameters': self.parameter_counts,
            'Size_KB': self.sizes_kb,
            'Sparsity_Percent': self.sparsities,
            'Accuracy_Percent': self.accuracies,
            'Inference_Time_ms': self.inference_times_ms
        })
        df.to_csv(filepath, index=False)
        print(f"\n ◱ ✓ Metrics saved to: {filepath}")




In [ ]:
# ============================================
# COMPRESSION STAGES 
# ============================================

def stage1_create_student(teacher_model, device, num_classes):
    """ Create smaller student model with correct output size"""
    print("\n STAGE 1: Creating Smaller Student Model (Layer Pruning)")
    
    # we used 16 hidden units because it is a common choice for small models and provides a good balance between capacity and size.
    #  It also allows us to easily copy weights from the teacher model's first 16 units, 
    # which can help the student model learn more effectively during distillation.

    # it can be changed to 64, the actual teacher model has 64 hidden units, but we want to create a smaller student model that can still learn effectively from the teacher.
    student = StudentGRU(input_size=7, hidden_size=16, num_classes=num_classes).to(device)
    
    # copy weights where possible
    # this excutes if the teacher model has the same or more hidden units than the student,(64) allowing us to initialize the student with a subset (prunned) of the teacher's weights, 
    # which can improve learning during distillation.

    try:
        with torch.no_grad():
            if hasattr(teacher_model.gru, 'weight_ih_l0'):
                if teacher_model.gru.weight_ih_l0.shape[0] >= student.gru.weight_ih_l0.shape[0]:
                    student.gru.weight_ih_l0.copy_(
                        teacher_model.gru.weight_ih_l0[:student.gru.weight_ih_l0.shape[0]]
                    )
                    student.gru.weight_hh_l0.copy_(
                        teacher_model.gru.weight_hh_l0[:student.gru.weight_hh_l0.shape[0]]
                    )
                    print("   ✓ Initialized student from teacher weights")
    except Exception as e:
        print(f"   🟠 Random initialization: {e}")
    
    teacher_params = sum(p.numel() for p in teacher_model.parameters())
    student_params = sum(p.numel() for p in student.parameters())
    
    print(f"   Teacher: {teacher_params:,} params → Student: {student_params:,} params")
    print(f"   Reduction: {(1 - student_params/teacher_params)*100:.1f}%")
    
    return student

def stage2_weight_pruning(model, pruning_amount=0.5):
    """Prune individual weights"""
    print(f"\n STAGE 2: Weight Pruning ({pruning_amount*100:.0f}% sparsity target)")
    
    model_copy = copy.deepcopy(model)
    
    for name, module in model_copy.named_modules():
        if isinstance(module, nn.GRU):
            prune.l1_unstructured(module, name='weight_ih_l0', amount=pruning_amount)
            prune.l1_unstructured(module, name='weight_hh_l0', amount=pruning_amount)
            prune.remove(module, 'weight_ih_l0')
            prune.remove(module, 'weight_hh_l0')
            
        elif isinstance(module, nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=pruning_amount * 0.3)
            prune.remove(module, 'weight')
    
    total_params = 0
    zero_params = 0
    for param in model_copy.parameters():
        total_params += param.numel()
        zero_params += (param == 0).sum().item()
    
    sparsity = (zero_params / total_params) * 100
    print(f"   ✓ Achieved {sparsity:.1f}% sparsity")
    print(f"  ✓ Non-zero parameters: {total_params - zero_params:,}")
    
    return model_copy


def stage3_knowledge_distillation(teacher_model, student_model, train_loader, 
                                   test_loader, device, epochs=20, temperature=4.0, alpha=0.7):
    """Distill knowledge from teacher to student"""
    print(f"\n STAGE 3: Knowledge Distillation")
    print(f"   Temperature: {temperature}, Alpha: {alpha}, Epochs: {epochs}")
    
    optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-3)
    
    def distillation_loss(student_logits, teacher_logits, true_labels):
        soft_student = nn.functional.log_softmax(student_logits / temperature, dim=1)
        soft_teacher = nn.functional.softmax(teacher_logits / temperature, dim=1)
        soft_loss = nn.functional.kl_div(soft_student, soft_teacher, reduction='batchmean')
        hard_loss = nn.functional.cross_entropy(student_logits, true_labels)
        return alpha * soft_loss * (temperature**2) + (1 - alpha) * hard_loss
    
    teacher_model.eval()
    best_acc = 0
    best_model = None
    
    for epoch in range(epochs):
        student_model.train()
        total_loss = 0
        
        for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            with torch.no_grad():
                teacher_logits = teacher_model(X_batch)
            
            student_logits = student_model(X_batch)
            loss = distillation_loss(student_logits, teacher_logits, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Evaluate
        if (epoch + 1) % 5 == 0:
            student_model.eval()
            all_preds = []
            all_labels = []
            with torch.no_grad():
                for X_batch, y_batch in test_loader:
                    X_batch = X_batch.to(device)
                    logits = student_model(X_batch)
                    preds = logits.argmax(dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_labels.extend(y_batch.numpy())
            acc = accuracy_score(all_labels, all_preds) * 100
            print(f"   Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%")
            
            if acc > best_acc:
                best_acc = acc
                best_model = copy.deepcopy(student_model)
    
    return best_model if best_model else student_model

import io

def stage4_quantization(model):
    """Quantize to INT8 (final deployment model)"""
    print("\n ⁙ STAGE 4: Quantization (32-bit → 8-bit)")
    
    original_device = next(model.parameters()).device
    print(f"   Original device: {original_device}")
    
    model.eval()
    model.to('cpu')
    
    quantized_model = torch.quantization.quantize_dynamic(
        model,
        {nn.GRU, nn.Linear},
        dtype=torch.qint8
    )
    
    # Measure size by serializing to a bytes buffer
    def get_model_size_kb(m):
        buffer = io.BytesIO()
        torch.save(m.state_dict(), buffer)
        return buffer.tell() / 1024
    
    original_size = get_model_size_kb(model)
    quantized_size = get_model_size_kb(quantized_model)
    
    print(f"   Size: {original_size:.2f} KB → {quantized_size:.2f} KB")
    print(f"   Reduction: {(1 - quantized_size/original_size)*100:.1f}%")
    print(f"   Final device: CPU (ready for deployment)")
    
    return quantized_model

In [ ]:
# ============================================
# MAIN COMPRESSION PIPELINE
# ============================================

def compress_full_pipeline(original_model, train_loader, test_loader, device, num_classes):
    """
    Run all four compression stages with metrics tracking.
    """
    print("="*80)
    print(" ✓ COMPLETE MODEL COMPRESSION PIPELINE")
    print("   Stage 1: Student (Layer Pruning) → Stage 2: Weight Pruning")
    print("   → Stage 3: Distillation → Stage 4: Quantization")
    print("="*80)
    
    metrics = ModelMetrics()
    
    # Baseline
    print("\n ↈ BASELINE: Original Teacher Model")
    metrics.log_stage("0. Original (Teacher)", original_model, test_loader, device)
    
    # Stage 1: Create student 
    student = stage1_create_student(original_model, device, num_classes)
    metrics.log_stage("1. Student (Layer Pruned)", student, test_loader, device)
    
    # Stage 2: Weight pruning
    pruned = stage2_weight_pruning(student, pruning_amount=0.5)
    metrics.log_stage("2. Weight Pruned", pruned, test_loader, device)
    
    # Stage 3: Distillation
    distilled = stage3_knowledge_distillation(original_model,pruned, train_loader, 
                                               test_loader, device, epochs=20)
    metrics.log_stage("3. Distilled", distilled, test_loader, device)
    
    # Stage 4: Quantization (moves to CPU)
    quantized = stage4_quantization(distilled)
    metrics.log_stage("4. Quantized (INT8)", quantized, test_loader, 'cpu')
    
    # Summary
    metrics.print_summary_table()
    metrics.save_to_file('compression_metrics.csv')
    
    return quantized, metrics



In [ ]:
# ============================================
# LOAD TRAINED MODEL AND DATA
# ============================================

def inspect_saved_model(model_path):
    """Inspect the saved model to get correct parameters"""
    print(f"\n Inspecting saved model: {model_path}")
    with h5py.File(model_path, 'r') as f:
        print("   Keys in saved model:")
        for key in f.keys():
            shape = f[key].shape
            print(f"      {key}: {shape}")
        
        # Get head layer shape to determine num_classes
        if 'head.weight' in f:
            head_shape = f['head.weight'].shape
            num_classes = head_shape[0]
            hidden_size = head_shape[1]
            print(f"\n    Detected: hidden_size={hidden_size}, num_classes={num_classes}")
            return hidden_size, num_classes
    
    return 64, 4  # defaults


def load_your_model(model_path, device):
    """Load the model with correct architecture based on saved file"""
    print(f"\n ◱ Loading model from: {model_path}")
    
    # inspect the saved model to get correct dimensions
    with h5py.File(model_path, 'r') as f:
        # Get head layer shape
        head_shape = f['head.weight'].shape
        num_classes = head_shape[0]
        hidden_size = head_shape[1]
        
        # Get GRU layer shape
        gru_weight_shape = f['gru.weight_ih_l0'].shape
        input_size = gru_weight_shape[1]  # should be 7
        num_layers = 2  # this is default from  training
        
        print(f"  ◘ Detected architecture:")
        print(f"      input_size: {input_size}")
        print(f"      hidden_size: {hidden_size}")
        print(f"      num_classes: {num_classes}")
        print(f"      num_layers: {num_layers}")
    
    # Create model with correct dimensions
    model = GRUClassifier(
        input_size=input_size,
        hidden_size=hidden_size,
        num_classes=num_classes,
        num_layers=num_layers
    )
    
    # Load state dict
    with h5py.File(model_path, 'r') as f:
        state_dict = {}
        for key in f.keys():
            state_dict[key] = torch.tensor(f[key][:])
    
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"   ✓ Loaded: {total_params:,} parameters")
    print(f"   Device: {next(model.parameters()).device}")
    
    return model, num_classes

In [ ]:
# ============================================
# EXECUTION - INTEGRATE WITH TRAINING
# ============================================

if __name__ == "__main__":
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🔧 Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
    
    # ========================================
    # Define Drive path for saving
    # ========================================
    DRIVE_DIR = '/content/drive/MyDrive/c/ai/traffic/uncompressed/'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    
    # ========================================
    # Load saved model and data
    # ========================================
    
    MODEL_PATH = os.path.join(DRIVE_DIR, 'gru_traffic_model.h5')
    
    if os.path.exists(MODEL_PATH):
        # Load model with automatic dimension detection
        teacher_model, num_classes = load_your_model(MODEL_PATH, DEVICE)
        
        # Load saved data from training
        data_path = os.path.join(DRIVE_DIR, 'training_data.npz')
        if os.path.exists(data_path):
            data = np.load(data_path, allow_pickle=True)
            train_X = data['train_X']
            train_y = data['train_y']
            test_X = data['test_X']
            test_y = data['test_y']
            
            from torch.utils.data import TensorDataset, DataLoader
            class SimpleDataset:
                def __init__(self, X, y):
                    self.X = torch.tensor(X, dtype=torch.float32)
                    self.y = torch.tensor(y, dtype=torch.long)
                def __len__(self): return len(self.y)
                def __getitem__(self, i): return self.X[i], self.y[i]
            
            train_loader = DataLoader(SimpleDataset(train_X, train_y), batch_size=128, shuffle=True)
            test_loader = DataLoader(SimpleDataset(test_X, test_y), batch_size=128, shuffle=False)
            
            print(f"   ✓ Loaded {len(train_X)} training samples, {len(test_X)} test samples")
            print(f"   ✓ Labels range: 0-{num_classes-1} (total {num_classes} classes)")
            
            # Run compression with correct num_classes
            final_model, metrics = compress_full_pipeline(teacher_model, train_loader, test_loader, DEVICE, num_classes)
            
            # ========================================
            # SAVE FULL QUANTIZED MODEL AND DICTS, NOT JUST STATE_DICT
            # ========================================
            # Save to  directory 
            torch.save(final_model, 'deployment_model_quantized_full.pt')
            print(f"\n ◱ Final compressed model saved locally: deployment_model_quantized_full.pt")
            
            # save to Google Drive
            drive_model_path = os.path.join(DRIVE_DIR, 'deployment_model_quantized_full.pt')
            torch.save(final_model, drive_model_path)
            print(f" ◱ Final compressed model saved to Drive: {drive_model_path}")
            
            # Save state_dict as backup (but we donot recommended for quantized) 
            torch.save(final_model.state_dict(), os.path.join(DRIVE_DIR, 'deployment_model_quantized_state.pt'))
            print(f" ◱ State dict backup saved to: {os.path.join(DRIVE_DIR, 'deployment_model_quantized_state.pt')}")
            
            # Print model info
            print(f"\n ↈ Model Info:")
            print(f"   Type: {type(final_model)}")
            print(f"   Size: {os.path.getsize('deployment_model_quantized_full.pt') / 1024:.2f} KB")
            
        else:
            print(f"\n   ✕ training_data.npz not found at: {data_path}")
    
    else:
        print(f"\n   ✕ Model not found at: {MODEL_PATH}")
        print("   Run your training script first to save the model.")

🔧 Device: cuda
   GPU: Tesla T4

📂 Loading model from: /content/drive/MyDrive/c/ai/traffic/uncompressed/gru_traffic_model.h5
   Detected architecture:
      input_size: 7
      hidden_size: 64
      num_classes: 3
      num_layers: 2
   ✅ Loaded: 39,171 parameters
   Device: cuda:0
   ✅ Loaded 32204 training samples, 8052 test samples
   ✅ Labels range: 0-2 (total 3 classes)
🎯 COMPLETE MODEL COMPRESSION PIPELINE
   Stage 1: Student (Layer Pruning) → Stage 2: Weight Pruning
   → Stage 3: Distillation → Stage 4: Quantization

📦 BASELINE: Original Teacher Model

📊 Evaluating: 0. Original (Teacher)

📈 Results for 0. Original (Teacher):
   Parameters:     39,171
   Model size:     155.59 KB (0.152 MB)
   Sparsity:       0.0%
   Accuracy:       96.56%
   Inference time: 0.008 ms per sample
   Device:         cuda:0

🔪 STAGE 1: Creating Smaller Student Model (Layer Pruning)
   ⚠️ Random initialization: The size of tensor a (16) must match the size of tensor b (64) at non-singleton dimension 1

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1415: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)
  result = _VF.gru(



📈 Results for 2. Weight Pruned:
   Parameters:     1,251
   Model size:     7.58 KB (0.007 MB)
   Sparsity:       44.7%
   Accuracy:       43.91%
   Inference time: 0.011 ms per sample
   Device:         cuda:0

📚 STAGE 3: Knowledge Distillation
   Temperature: 4.0, Alpha: 0.7, Epochs: 20


Epoch 1/20:   0%|          | 0/252 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1415: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)
  result = _VF.gru(
Epoch 5/20: 100%|██████████| 252/252 [00:00<00:00, 257.09it/s]


   Epoch 5 | Loss: 0.5205 | Acc: 91.95%


...

   Epoch 15 | Loss: 0.1774 | Acc: 95.88%


Epoch 16/20:   0%|          | 0/252 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1415: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)
  result = _VF.gru(
Epoch 20/20: 100%|██████████| 252/252 [00:00<00:00, 263.31it/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1415: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1480.)
  result = _VF.gru(


   Epoch 20 | Loss: 0.1444 | Acc: 95.74%

📊 Evaluating: 3. Distilled

📈 Results for 3. Distilled:
   Parameters:     1,251
   Model size:     7.58 KB (0.007 MB)
   Sparsity:       0.0%
   Accuracy:       95.88%
   Inference time: 0.010 ms per sample
   Device:         cuda:0

🔢 STAGE 4: Quantization (32-bit → 8-bit)
   Original device: cuda:0
   Size: 7.58 KB → 5.26 KB
   Reduction: 30.6%
   Final device: CPU (ready for deployment)

📊 Evaluating: 4. Quantized (INT8)


/tmp/ipykernel_9740/2633413597.py:142: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(



📈 Results for 4. Quantized (INT8):
   Parameters:     0
   Model size:     5.26 KB (0.005 MB)
   Sparsity:       0.0%
   Accuracy:       95.81%
   Inference time: 0.021 ms per sample
   Device:         cpu

📊 COMPRESSION PIPELINE SUMMARY
Stage                Params       Size(KB)     Sparsity   Accuracy     Time(ms)    
------------------------------------------------------------------------------------------
0. Original (Teacher) 39,171       155.59       0.0       % 96.56       % 0.008       
1. Student (Layer Pruned) 1,251        7.16         0.0       % 38.62       % 0.008       
2. Weight Pruned     1,251        7.58         44.7      % 43.91       % 0.011       
3. Distilled         1,251        7.58         0.0       % 95.88       % 0.010       
4. Quantized (INT8)  0            5.26         0.0       % 95.81       % 0.021       
------------------------------------------------------------------------------------------

📈 FINAL RESULTS:
   Total Compression:  96.6% smaller
   A

In [ ]:
import os

# Define the drive folder used for the teacher model
drive_dir = '/content/drive/MyDrive/c/ai/traffic/compressed/'
os.makedirs(drive_dir, exist_ok=True)   # ensuring folder exists

# Save the quantized model
torch.save(final_model, os.path.join(drive_dir, 'deployment_model_quantized.pt'))

print(f"✓ Compressed model saved to: {os.path.join(drive_dir, 'deployment_model_quantized.pt')}")

✅ Compressed model saved to: /content/drive/MyDrive/c/ai/traffic/compressed/deployment_model_quantized.pt


In [ ]:
import time
import torch
import numpy as np

# ============================================
# Load the FULL model but not just state_dict
# ============================================
device = torch.device('cpu')

# load it directly since saved with torch.save
model = torch.load(
    '/content/drive/MyDrive/c/ai/deployment_model_quantized.pt', 
    map_location=device,
    weights_only=False
)
model.eval()
# Note: Quantized models don't have traditional parameters, so we avoided calling .parameters()

print("✓ Model loaded successfully")
print(f"   Model type: {type(model)}")

# ============================================
# Test inference
# ============================================
dummy_input = torch.randn(1, 16, 7)
with torch.no_grad():
    output = model(dummy_input)
print(f"   Test output shape: {output.shape}")

# ============================================
# Load test data and measure latency
# ============================================
data = np.load('/content/drive/MyDrive/c/ai/traffic/uncompressed/training_data.npz', allow_pickle=True)
test_X = data['test_X']
test_y = data['test_y']
print(f"   Test samples: {len(test_X)}")

# Convert to tensor
test_tensor = torch.tensor(test_X, dtype=torch.float32)

# Measure latency
latencies = []
with torch.no_grad():
    # Warm-up
    for _ in range(5):
        _ = model(test_tensor[:1])
    
    # Measure all test samples in batches
    batch_size = 128
    for i in range(0, len(test_tensor), batch_size):
        batch = test_tensor[i:i+batch_size]
        start = time.perf_counter()
        _ = model(batch)
        end = time.perf_counter()
        latencies.append((end - start) / len(batch))

latencies_ms = [l * 1000 for l in latencies]
print(f"\n ↝ Inference Latency on CPU (ThinkPad T460):")
print(f"   Mean: {np.mean(latencies_ms):.3f} ms")
print(f"   Std:  {np.std(latencies_ms):.3f} ms")
print(f"   Min:  {np.min(latencies_ms):.3f} ms")
print(f"   Max:  {np.max(latencies_ms):.3f} ms")
print(f"   Throughput: {1000 / np.mean(latencies_ms):.0f} samples/sec")

✅ Model loaded successfully
   Model type: <class '__main__.StudentGRU'>
   Test output shape: torch.Size([1, 3])
   Test samples: 8052

📊 Inference Latency on CPU (ThinkPad T460):
   Mean: 0.014 ms
   Std:  0.002 ms
   Min:  0.011 ms
   Max:  0.024 ms
   Throughput: 72524 samples/sec


/usr/local/lib/python3.12/dist-packages/torch/_utils.py:445: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


## WEATHER GRU NOTEBOOKS

In [ ]:
# uncomment this, run this cell alone, ater istaltion,commenthis line out, then refresh the kernel, do this for every session only once

# !pip install tensorflow-model-optimization
# !pip install tf-keras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import tensorflow as tf
import tf_keras as keras_v2
import tensorflow_model_optimization as tfmot
import time, os, tempfile, io, sys
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# optional imports for tracking and visualization
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    
# optional import for progress bars
try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    tqdm = lambda x, **kw: x

# optional import for plotting
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

# Ensure we use the legacy Keras API for compatibility with TensorFlow 2.x
os.environ["TF_USE_LEGACY_KERAS"] = "1"


In [ ]:
class ModelMetrics:
    def __init__(self):
        self.stages, self.sizes_kb, self.accuracies = [], [], []
        self.inference_times_ms, self.sparsities, self.parameter_counts = [], [], []
        self.classification_reports = []   # store dict reports
        self.mem_usage_mb = []             # peak memory during eval (optional)

    def _get_memory_usage(self):
        if HAS_PSUTIL:
            return psutil.Process().memory_info().rss / (1024*1024)
        return np.nan

    def log_stage(self, stage_name, model, x_test, y_test, is_tflite=False):
        print(f"\n{'='*60}\n ↝ Evaluating: {stage_name}\n{'='*60}")
        # Memory before eval
        mem_before = self._get_memory_usage()
        size_kb = len(model)/1024 if is_tflite else self._keras_size(model)
        sparsity = 0
        if not is_tflite:
            ws = model.get_weights()
            total = sum(w.size for w in ws)
            zeros = sum(np.count_nonzero(w == 0) for w in ws)
            sparsity = zeros/total*100 if total else 0
        # Latency & predictions
        latencies, y_pred = [], []
        if is_tflite:
            interp = tf.lite.Interpreter(model_content=model)
            interp.allocate_tensors()
            inp = interp.get_input_details()[0]
            out = interp.get_output_details()[0]
            for i in range(len(x_test)):
                interp.set_tensor(inp['index'], x_test[i:i+1].astype(np.float32))
                t0 = time.perf_counter()
                interp.invoke()
                latencies.append(time.perf_counter() - t0)
                y_pred.append(np.argmax(interp.get_tensor(out['index']), axis=1)[0])
            avg_ms = np.mean(latencies) * 1000
            # Percentiles for latency distribution
            lat_50 = np.percentile(latencies, 50)*1000
            lat_95 = np.percentile(latencies, 95)*1000
            print(f"Latency (ms) - Mean: {avg_ms:.3f}, 50th: {lat_50:.3f}, 95th: {lat_95:.3f}")
        else:
            t0 = time.perf_counter()
            preds = model.predict(x_test, verbose=0)
            avg_ms = (time.perf_counter()-t0)/len(x_test)*1000
            y_pred = np.argmax(preds, axis=1)
        # Accuracy & Classification Report
        acc = accuracy_score(y_test, y_pred)*100
        clf_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        self.classification_reports.append(clf_report)
        # Memory after eval
        mem_after = self._get_memory_usage()
        mem_delta = mem_after - mem_before if not np.isnan(mem_before) else 0
        params = model.count_params() if not is_tflite else 0
        self.stages.append(stage_name)
        self.sizes_kb.append(size_kb)
        self.accuracies.append(acc)
        self.inference_times_ms.append(avg_ms)
        self.sparsities.append(sparsity)
        self.parameter_counts.append(params)
        self.mem_usage_mb.append(mem_delta)
        print(f"Results: {acc:.2f}% Acc | {size_kb:.2f} KB | {avg_ms:.3f} ms/sample")
        if sparsity:
            print(f"Sparsity: {sparsity:.1f}% zeros")
        if not np.isnan(mem_delta):
            print(f"Memory Δ: {mem_delta:.2f} MB")
        # Optionally show confusion matrix if matplotlib is available
        if HAS_MPL and not is_tflite and len(np.unique(y_test)) <= 10:
            cm = confusion_matrix(y_test, y_pred)
            plt.figure(figsize=(6,5))
            plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            plt.title(f'Confusion Matrix - {stage_name}')
            plt.colorbar()
            plt.ylabel('True label')
            plt.xlabel('Predicted label')
            plt.show()
        return acc, size_kb, avg_ms

    def _keras_size(self, model):
        with tempfile.NamedTemporaryFile(suffix='.keras', delete=True) as tmp:
            model.save(tmp.name)
            return os.path.getsize(tmp.name)/1024

    def print_summary_table(self):
        print("\n"+"="*90)
        print(f"{'Stage':<22}{'Params':<12}{'Size(KB)':<12}{'Sparsity':<10}{'Accuracy':<12}{'Time(ms)':<10}{'Mem Δ(MB)'}")
        for i,s in enumerate(self.stages):
            mem_str = f"{self.mem_usage_mb[i]:.1f}" if not np.isnan(self.mem_usage_mb[i]) else "N/A"
            print(f"{s:<22}{self.parameter_counts[i]:<12,}{self.sizes_kb[i]:<12.2f}"
                  f"{self.sparsities[i]:<10.1f}%{self.accuracies[i]:<12.2f}%{self.inference_times_ms[i]:<10.3f}{mem_str}")

    def save_to_file(self, path):
        # Build comprehensive DataFrame
        df = pd.DataFrame({
            'Stage': self.stages,
            'Params': self.parameter_counts,
            'Size_KB': self.sizes_kb,
            'Sparsity_%': self.sparsities,
            'Accuracy_%': self.accuracies,
            'Inference_ms': self.inference_times_ms,
            'Mem_Delta_MB': self.mem_usage_mb
        })
        # Also store per-class metrics from classification reports (optional)
        for i, report in enumerate(self.classification_reports):
            for class_label, metrics in report.items():
                if isinstance(metrics, dict):
                    for metric_name, val in metrics.items():
                        df.loc[i, f'{class_label}_{metric_name}'] = val
        # Save to CSV using BytesIO first (in-memory)
        buffer = io.BytesIO()
        df.to_csv(buffer, index=False)
        buffer.seek(0)
        # Write to file from buffer
        with open(path, 'wb') as f:
            f.write(buffer.getvalue())
        print(f" ✓ Metrics saved to {path} (size: {buffer.getbuffer().nbytes} bytes)")


In [ ]:
#============================================
# MODEL ARCHITECTURE (from training)
# ============================================

def build_teacher(input_shape, num_classes):
    inputs = keras_v2.Input(shape=input_shape)
    x = keras_v2.layers.GRU(64, return_sequences=True, activation='tanh',
                             unroll=True,
                             kernel_regularizer=keras_v2.regularizers.l2(0.001))(inputs)
    x = keras_v2.layers.Dropout(0.3)(x)
    x = keras_v2.layers.BatchNormalization()(x)
    x = keras_v2.layers.GRU(32, return_sequences=False, activation='tanh',
                             unroll=True,
                             kernel_regularizer=keras_v2.regularizers.l2(0.001))(x)
    x = keras_v2.layers.Dropout(0.3)(x)
    x = keras_v2.layers.Dense(16, activation='relu')(x)
    x = keras_v2.layers.Dropout(0.2)(x)
    outputs = keras_v2.layers.Dense(num_classes, activation='softmax')(x)
    return keras_v2.Model(inputs, outputs)

def build_student(input_shape, num_classes):
    inputs = keras_v2.Input(shape=input_shape)
    x = keras_v2.layers.GRU(32, return_sequences=False, activation='tanh',
                             unroll=True)(inputs)
    x = keras_v2.layers.Dropout(0.2)(x)
    x = keras_v2.layers.Dense(16, activation='relu')(x)
    outputs = keras_v2.layers.Dense(num_classes, activation='softmax')(x)
    return keras_v2.Model(inputs, outputs)



In [ ]:
# ============================================
# COMPRESSION STAGES
# ============================================

def stage2_weight_pruning(model, train_X, train_y, val_X, val_y, epochs=5, pruning_amount=0.5):
    print(f"\n STAGE 2: Weight Pruning ({pruning_amount*100:.0f}% target)")
    pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0, final_sparsity=pruning_amount,
        begin_step=0, end_step=(len(train_X) // 32) * epochs
    )
    pruned = tfmot.sparsity.keras.prune_low_magnitude(
        model, pruning_params={'pruning_schedule': pruning_schedule}
    )
    pruned.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    pruned.fit(train_X, train_y, validation_data=(val_X, val_y),
               epochs=epochs, batch_size=32,
               callbacks=[tfmot.sparsity.keras.UpdatePruningStep()], verbose=1)
    return tfmot.sparsity.keras.strip_pruning(pruned)

def stage3_knowledge_distillation(teacher, student, train_X, train_y, val_X, val_y,
                                   epochs=20, temperature=4.0, alpha=0.7):
    print(f"\n  STAGE 3: Knowledge Distillation (epochs={epochs})")
    student.compile(optimizer=keras_v2.optimizers.Adam(1e-3),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    optimizer = keras_v2.optimizers.Adam(1e-3)

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            s_logits = student(x, training=True)
            t_logits = teacher(x, training=False)
            log_soft_s = tf.nn.log_softmax(s_logits / temperature)
            soft_t = tf.nn.softmax(t_logits / temperature)
            soft_loss = tf.reduce_mean(tf.reduce_sum(
                soft_t * (tf.nn.log_softmax(t_logits / temperature) - log_soft_s), axis=1))
            hard_loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(y, s_logits))
            loss = alpha * soft_loss * (temperature**2) + (1 - alpha) * hard_loss
        grads = tape.gradient(loss, student.trainable_variables)
        optimizer.apply_gradients(zip(grads, student.trainable_variables))
        return loss

    dataset = tf.data.Dataset.from_tensor_slices((train_X, train_y)).batch(32)
    for epoch in tqdm(range(epochs), desc="Distillation Epochs", disable=not HAS_TQDM):
        for xb, yb in dataset:
            train_step(xb, yb)
        if (epoch + 1) % 5 == 0:
            acc = student.evaluate(val_X, val_y, verbose=0)[1]
            print(f"Epoch {epoch+1}/{epochs} | Val Acc: {acc:.4f}")
    return student

def stage4_quantization(model, representative_dataset):
    print("\n STAGE 4: Quantization")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset

    # Use TFLITE_BUILTINS instead of TFLITE_BUILTINS_INT8
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
    
    # Set to False to avoid Flex ops
    converter._experimental_lower_tensor_list_ops = False
    converter.inference_input_type = tf.float32
    converter.inference_output_type = tf.float32
    return converter.convert()


In [ ]:
# ============================================
# COMPRESSION PIPELINE
# ============================================

# make representative dataset generator for quantization calibration
def make_representative_dataset(train_X, num_samples=100, seed=42):
    np.random.seed(seed)
    indices = np.random.choice(len(train_X), num_samples, replace=False)
    def gen():
        for idx in indices:
            yield [train_X[idx:idx+1].astype(np.float32)]
    return gen

# compress the model through all stages and log metrics at each stage
def compress_weather_pipeline(teacher, train_X, train_y, val_X, val_y, test_X, test_y,
                             input_shape, num_classes, verbose=True):
    metrics = ModelMetrics()
    
    metrics.log_stage("0. Teacher (Original)", teacher, test_X, test_y, is_tflite=False)
    
    # Student model
    student = build_student(input_shape, num_classes)
    metrics.log_stage("1. Student (Baseline)", student, test_X, test_y, is_tflite=False)
    
    # Weight pruning
    pruned_student = stage2_weight_pruning(student, train_X, train_y, val_X, val_y, 
                                          epochs=5, pruning_amount=0.5)
    metrics.log_stage("2. Student (Weight Pruned)", pruned_student, test_X, test_y, is_tflite=False)
    
    # Knowledge distillation
    distilled_student = stage3_knowledge_distillation(teacher, pruned_student, 
                                                     train_X, train_y, val_X, val_y, 
                                                     epochs=20)
    metrics.log_stage("3. Student (Distilled)", distilled_student, test_X, test_y, is_tflite=False)
    
    # Quantization
    rep_dataset = make_representative_dataset(train_X, num_samples=100)
    quantized_model = stage4_quantization(distilled_student, rep_dataset)
    metrics.log_stage("4. Quantized (Float32 IO)", quantized_model, test_X, test_y, is_tflite=True)
    
    # print final summary
    metrics.print_summary_table()

    # save metrics to CSV in Drive
    file_path = '/content/drive/MyDrive/c/ai/weather/compressed-gru/'
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    metrics.save_to_file(file_path + 'weather_compression_metrics.csv')
    
    return quantized_model, metrics

In [ ]:
if __name__ == "__main__":
    # Load  data and model
    try:
        # load data sequesnces
        data = np.load('/content/drive/MyDrive/c/ai/weather/uncompressed-gru/weather_sequence_data.npz')
        X_train, y_train = data['X_train'], data['y_train']
        X_val, y_val = data['X_val'], data['y_val']
        X_test, y_test = data['X_test'], data['y_test']
        input_shape = X_train.shape[1:]
        num_classes = len(np.unique(y_train))

        print(f" ✓ Data loaded. Input Shape: {input_shape}, Classes: {num_classes}")

    except FileNotFoundError:
        print(" ✖ Error: 'weather_sequence_data.npz' not found. Please check your path.")
        raise

    # Build and load teacher weights
    teacher = build_teacher(input_shape, num_classes)
    teacher.load_weights('/content/drive/MyDrive/c/ai/weather/uncompressed-gru/gru_weather_model.h5')
    teacher.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(" ✓ Teacher weights loaded and compiled.")

    # Run pipeline
    final_tflite_model, metrics = compress_weather_pipeline(
        teacher, X_train, y_train, X_val, y_val, X_test, y_test,
        input_shape, num_classes
    )

    
    # Save final quantized model
    save_path = '/content/drive/MyDrive/c/ai/weather/compressed-gru/weather_model_quantized.tflite'

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    print(" ✓ Created directory for saving quantized model if it didn't exist.")
    
    with open(save_path, 'wb') as f:
        f.write(final_tflite_model)
    
    print(f" ✓ Quantized model saved to: {save_path}")


In [ ]:
# ============================================
# INFERENCE LATENCY MEASUREMENT
# ============================================

import time
import numpy as np
import tensorflow as tf

# Load the quantized model
model_path = '/content/drive/MyDrive/c/ai/weather/compressed-gru/weather_model_quantized.tflite' 
 
# Initialize the Interpreter
# we avoided the use of TFLITE_BUILTINS_INT8 in the converter, so we can load it with the standard 
# Interpreter without needing to handle Flex ops like delegates or custom ops e.g. libflex.so

# interpreter = tf.lite.Interpreter(
#     model_path=model_path,
#     experimental_delegates=[tf.lite.experimental.load_delegate('libflex_delegate.so')]
# )
# interpreter.allocate_tensors()

interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# Load test data
data = np.load('/content/drive/MyDrive/c/ai/weather/uncompressed-gru/weather_sequence_data.npz', allow_pickle=True)
test_X = data['X_test'].astype(np.float32)

# Measure latency
latencies = []
num_samples = min(1000, len(test_X))

for i in range(num_samples):
    sample = test_X[i:i+1]
    
    # No quantization scaling needed (model uses float32 input/output)
    start = time.perf_counter()
    interpreter.set_tensor(input_details['index'], sample)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details['index'])
    end = time.perf_counter()
    latencies.append(end - start)

latencies_ms = [l * 1000 for l in latencies]

print(f"\nInference Latency on CPU (Weather Model):")
print(f"   Mean: {np.mean(latencies_ms):.3f} ms")
print(f"   Std:  {np.std(latencies_ms):.3f} ms")
print(f"   Min:  {np.min(latencies_ms):.3f} ms")
print(f"   Throughput: {1000 / np.mean(latencies_ms):.0f} samples/sec")


Inference Latency on CPU (Weather Model):
   Mean: 0.043 ms
   Std:  0.026 ms
   Min:  0.033 ms
   Throughput: 23076 samples/sec


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
